# Complex multi-agent system with Pydantic AI

This lab builds a research agent that combines multiple patterns: orchestrator-workers, parallelization, and tool use. The agent conducts interviews by searching the web, then synthesizes findings into a report.

```mermaid
flowchart TB
    subgraph Research Agent
        Start([Topic]) --> Orchestrator
        Orchestrator -->|"Analyst 1"| Interview1["Interview Agent 1"]
        Orchestrator -->|"Analyst 2"| Interview2["Interview Agent 2"]
        Orchestrator -->|"Analyst N"| InterviewN["Interview Agent N"]
    end

    subgraph Interview Agent
        AskQ["Ask Question"] --> Search["Search Web + Wikipedia"]
        Search --> Answer["Generate Answer"]
        Answer -->|"More questions"| AskQ
        Answer -->|"Done"| WriteSection["Write Section"]
    end

    subgraph Report Generation
        Interview1 --> Intro["Write Introduction"]
        Interview2 --> Report["Write Report Body"]
        InterviewN --> Conclusion["Write Conclusion"]
        Intro --> Final["Finalize Report"]
        Report --> Final
        Conclusion --> Final
    end

    Final --> Output([Final Report])
```

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import asyncio
import os

from dotenv import load_dotenv
from jinja2 import Template
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessage

load_dotenv()


def load_prompt(prompt_filename, variables=None):
    variables = variables or {}
    with open(prompt_filename, "r") as f:
        template = Template(f.read())
        return template.render(**variables)

## Interview Agent

The interview agent asks questions, searches for information, generates expert answers, and writes a section summary.

In [ ]:
class SearchQuery(BaseModel):
    search_query: str = Field(description="Search query for retrieval.")


# Agents
question_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=load_prompt("assets/15_complex_agent/analyst_system_prompt.jinja2"),
)

search_query_agent = Agent(
    "openai:gpt-5-mini",
    output_type=SearchQuery,
    system_prompt=load_prompt("assets/15_complex_agent/search_system_prompt.jinja2"),
)

section_writer = Agent(
    "openai:gpt-5-mini",
    system_prompt=load_prompt(
        "assets/15_complex_agent/section_writer_system_prompt.jinja2"
    ),
)

In [ ]:
from serpapi import GoogleSearch


def search_web(query: str) -> str:
    """Search the web using SerpAPI."""
    params = {
        "q": query,
        "hl": "en",
        "google_domain": "google.com",
        "api_key": os.getenv("SERPAPI_API_KEY"),
    }
    search = GoogleSearch(params)
    results = search.get_dict()
    organic_results = results.get("organic_results", [])
    return "\n\n---\n\n".join(
        [
            f'<Document source="{doc["link"]}" page="{doc["title"]}"/>\n{doc["snippet"]}\n</Document>'
            for doc in organic_results
        ]
    )


def search_wikipedia(query: str) -> str:
    """Search Wikipedia."""
    from langchain_community.document_loaders import WikipediaLoader

    search_docs = WikipediaLoader(query=query, load_max_docs=2).load()
    return "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )

In [ ]:
async def conduct_interview(topic: str, max_turns: int = 2) -> dict:
    """Conduct a research interview on a topic."""
    message_history: list[ModelMessage] = []
    all_context = []

    # Initial question
    current_prompt = f"So you said you were writing an article on {topic}?"

    for turn in range(max_turns):
        # 1. Ask a question
        question_result = question_agent.run_sync(
            current_prompt, message_history=message_history
        )
        question = question_result.output
        message_history = question_result.all_messages()

        # 2. Generate search query
        search_result = search_query_agent.run_sync(question)
        query = search_result.output.search_query

        # 3. Search web and Wikipedia in parallel
        web_task = asyncio.to_thread(search_web, query)
        wiki_task = asyncio.to_thread(search_wikipedia, query)
        web_results, wiki_results = await asyncio.gather(web_task, wiki_task)

        context = f"{web_results}\n\n{wiki_results}"
        all_context.append(context)

        # 4. Generate expert answer
        expert_system = load_prompt(
            "assets/15_complex_agent/expert_system_prompt.jinja2",
            {"context": all_context},
        )
        expert_agent = Agent("openai:gpt-5-mini", system_prompt=expert_system)
        answer_result = expert_agent.run_sync(question, message_history=message_history)
        current_prompt = answer_result.output
        message_history = answer_result.all_messages()

    # 5. Write section
    section_result = section_writer.run_sync(
        f"Use this source to write your section: {all_context}"
    )

    return {
        "section": section_result.output,
        "context": all_context,
    }

## Research Agent

The research agent runs multiple interviews in parallel and synthesizes findings into a final report.

In [ ]:
intro_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a skilled report writer. Write a compelling introduction for a research report.",
)

report_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a skilled report writer. Write the main body of a research report based on the provided sections.",
)

conclusion_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a skilled report writer. Write a conclusion for a research report.",
)


async def run_research(topic: str, num_analysts: int = 3) -> str:
    """Run the full research pipeline."""
    # 1. Conduct interviews in parallel
    interview_tasks = [
        conduct_interview(topic, max_turns=2) for _ in range(num_analysts)
    ]
    interview_results = await asyncio.gather(*interview_tasks)

    sections = [r["section"] for r in interview_results]
    formatted_sections = "\n\n".join(sections)

    # 2. Write report parts in parallel
    intro_task = intro_agent.run(
        f"Write an introduction for a report about {topic}. Sections:\n{formatted_sections}"
    )
    body_task = report_agent.run(
        f"Write the main report about {topic} based on these sections:\n{formatted_sections}"
    )
    conclusion_task = conclusion_agent.run(
        f"Write a conclusion for a report about {topic}. Sections:\n{formatted_sections}"
    )

    intro_result, body_result, conclusion_result = await asyncio.gather(
        intro_task, body_task, conclusion_task
    )

    # 3. Finalize report
    final_report = (
        intro_result.output
        + "\n\n---\n\n"
        + body_result.output
        + "\n\n---\n\n"
        + conclusion_result.output
    )

    return final_report

In [ ]:
report = await run_research("Pydantic AI", num_analysts=3)
print(report)

## Exercise

Extend the research agent to include:
1. An evaluator that checks the quality of each section before including it in the report
2. A reviewer that checks the final report for consistency and quality